In [1]:
# Fetch the SRGAN code
!git clone https://github.com/Lornatang/SRGAN-PyTorch.git

# Move into the folder
%cd SRGAN-PyTorch

# Note: We are skipping the requirements.txt because Kaggle 
# already has a perfectly optimized PyTorch environment built-in!

Cloning into 'SRGAN-PyTorch'...
remote: Enumerating objects: 3631, done.
remote: Counting objects: 100% (920/920), done.
remote: Compressing objects: 100% (283/283), done.
remote: Total 3631 (delta 690), reused 810 (delta 633), pack-reused 2711 (from 1)
Receiving objects: 100% (3631/3631), 2.98 MiB | 18.70 MiB/s, done.
Resolving deltas: 100% (2385/2385), done.
/kaggle/working/SRGAN-PyTorch


In [2]:
import os
import yaml

# Hard reset to the correct base directory
os.chdir('/kaggle/working/SRGAN-PyTorch')
config_path = './configs/train/SRGAN_x4-ImageNet.yaml'

with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# 1. THE 90 EPOCH LOCK
cfg['TRAIN']['HYP']['EPOCHS'] = 90
cfg['TRAIN']['LR_SCHEDULER']['MILESTONES'] = [45, 68]

# 2. Link your SAR datasets (Base Path)
base_path = "/kaggle/input/datasets/maryclauds/sar-image-patches/SAR_Dataset/SAR_Dataset"

# 3. Fix the TRAIN paths
cfg['TRAIN']['DATASET']['TRAIN_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TRAIN']['DATASET']['TRAIN_LR_IMAGES_DIR'] = f"{base_path}/train_LR"

# 4. Fix the TEST paths (This stops the crash!)
cfg['TEST']['DATASET']['PAIRED_TEST_GT_IMAGES_DIR'] = f"{base_path}/train_HR"
cfg['TEST']['DATASET']['PAIRED_TEST_LR_IMAGES_DIR'] = f"{base_path}/train_LR"

# 5. Point to Phase 1 Weights
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_G_MODEL'] = "/kaggle/input/datasets/maryclauds/sar-image-patches/epoch_90.pth.tar"
cfg['TRAIN']['CHECKPOINT']['RESUMED_G_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['RESUMED_D_MODEL'] = ""

# 6. Fix Kaggle GPU flags
cfg['MODEL']['G']['COMPILED'] = False
cfg['MODEL']['D']['COMPILED'] = False
cfg['MODEL']['EMA']['COMPILED'] = False

with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

print(f"✅ Configuration locked! Target set precisely to: {cfg['TRAIN']['HYP']['EPOCHS']} epochs.")
print("✅ Test dataset paths successfully rerouted to your SAR dataset.")

✅ Configuration locked! Target set precisely to: 90 epochs.
✅ Test dataset paths successfully rerouted to your SAR dataset.


In [3]:
!python train_gan.py --config_path ./configs/train/SRGAN_x4-ImageNet.yaml

2026-03-16 21:51:04.830830: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773697865.007502      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773697865.060173      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773697865.486961      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773697865.487013      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773697865.487017      55 computation_placer.cc:177] computation placer alr

In [4]:
# Delete epochs 1 through 65 to free up Kaggle disk space
!for i in {1..65}; do rm -f /kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_$i.pth.tar; done

# Check how much space we have left now
!df -h /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.4G   18G  13% /kaggle/working


In [5]:
!ls -lh /kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/ | tail -n 10

total 2.1G
-rw-r--r-- 1 root root 270M Mar 17 00:13 epoch_66.pth.tar
-rw-r--r-- 1 root root 270M Mar 17 00:15 epoch_67.pth.tar
-rw-r--r-- 1 root root 270M Mar 17 00:17 epoch_68.pth.tar
-rw-r--r-- 1 root root 270M Mar 17 00:19 epoch_69.pth.tar
-rw-r--r-- 1 root root 270M Mar 17 00:21 epoch_70.pth.tar
-rw-r--r-- 1 root root 270M Mar 17 00:23 epoch_71.pth.tar
-rw-r--r-- 1 root root 270M Mar 17 00:25 epoch_72.pth.tar
-rw-r--r-- 1 root root 253M Mar 17 00:27 epoch_73.pth.tar


In [6]:
import os
import yaml

# 1. Force directory
os.chdir('/kaggle/working/SRGAN-PyTorch')
config_path = './configs/train/SRGAN_x4-ImageNet.yaml'

# 2. Delete the corrupted Epoch 73 file to free up space and prevent errors
corrupted_file = "/kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_73.pth.tar"
if os.path.exists(corrupted_file):
    os.remove(corrupted_file)
    print("🗑️ Corrupted Epoch 73 deleted.")

# 3. Open config
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# 4. Lock in the 90 limit
cfg['TRAIN']['HYP']['EPOCHS'] = 90

# 5. Tell the script to RESUME using your completely safe Epoch 72 master files!
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_G_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_D_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['RESUMED_G_MODEL'] = "/kaggle/working/SRGAN-PyTorch/results/SRGAN_x4-ImageNet/g_last.pth.tar"
cfg['TRAIN']['CHECKPOINT']['RESUMED_D_MODEL'] = "/kaggle/working/SRGAN-PyTorch/results/SRGAN_x4-ImageNet/d_last.pth.tar"

# 6. Save config
with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

print(f"✅ Configuration locked! Ready to resume from Epoch 72 and finish the final {90-72} epochs.")

🗑️ Corrupted Epoch 73 deleted.
✅ Configuration locked! Ready to resume from Epoch 72 and finish the final 18 epochs.


In [7]:
import os
import yaml

# Ensure we are in the right folder
os.chdir('/kaggle/working/SRGAN-PyTorch')
config_path = './configs/train/SRGAN_x4-ImageNet.yaml'

with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# The exact path to the surviving Epoch 72 file
safe_epoch_file = "/kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_72.pth.tar"

# Point BOTH resume variables to this massive combined file
cfg['TRAIN']['CHECKPOINT']['RESUMED_G_MODEL'] = safe_epoch_file
cfg['TRAIN']['CHECKPOINT']['RESUMED_D_MODEL'] = safe_epoch_file

# Make sure PRETRAINED is empty so it doesn't get confused
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_G_MODEL'] = ""
cfg['TRAIN']['CHECKPOINT']['PRETRAINED_D_MODEL'] = ""

with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

print("✅ Configuration locked! Bypassing the empty 'results' folder and pointing directly to the safe Epoch 72 backup.")

✅ Configuration locked! Bypassing the empty 'results' folder and pointing directly to the safe Epoch 72 backup.


In [8]:
!python train_gan.py --config_path ./configs/train/SRGAN_x4-ImageNet.yaml

2026-03-17 00:27:49.359254: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773707269.384354     169 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773707269.391268     169 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773707269.409137     169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773707269.409167     169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773707269.409170     169 computation_placer.cc:177] computation placer alr

In [9]:
import glob
import os

print("🔍 Scanning for ALL checkpoint files (No Filters)...\n")

# Find every single .pth.tar file anywhere in the working directory
all_files = glob.glob('/kaggle/working/**/*.pth.tar', recursive=True)

# Also check for standard .pth or .pt files just in case
all_files += glob.glob('/kaggle/working/**/*.pth', recursive=True)
all_files += glob.glob('/kaggle/working/**/*.pt', recursive=True)

for f in sorted(list(set(all_files))): # set() removes duplicates
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"[{size_mb:.1f} MB] {f}")

🔍 Scanning for ALL checkpoint files (No Filters)...

[269.7 MB] /kaggle/working/SRGAN-PyTorch/results/SRGAN_x4-ImageNet/d_best.pth.tar
[23.9 MB] /kaggle/working/SRGAN-PyTorch/results/SRGAN_x4-ImageNet/g_best.pth.tar
[269.7 MB] /kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_66.pth.tar
[269.7 MB] /kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_67.pth.tar
[269.7 MB] /kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_68.pth.tar
[269.7 MB] /kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_69.pth.tar
[269.7 MB] /kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_70.pth.tar
[269.7 MB] /kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_71.pth.tar
[269.7 MB] /kaggle/working/SRGAN-PyTorch/samples/SRGAN_x4-ImageNet/epoch_72.pth.tar


In [10]:
import os

# Ensure we are in the right folder
os.chdir('/kaggle/working/SRGAN-PyTorch')
file_path = 'utils.py'

# Read the original source code
with open(file_path, 'r') as f:
    source_code = f.read()

# 1. Patch the scheduler bug: If it asks for the scheduler, just tell it to 'pass' (ignore it)
source_code = source_code.replace(
    'scheduler.load_state_dict(checkpoint["scheduler"])', 
    'pass # [HOTFIX] Bypassed missing scheduler key!'
)

# 2. Preemptively patch the EMA bug (just in case they forgot to save that too)
source_code = source_code.replace(
    'ema_model.load_state_dict(checkpoint["ema_state_dict"])',
    'ema_model.load_state_dict(checkpoint.get("ema_state_dict", checkpoint["state_dict"])) # [HOTFIX] Fallback to standard weights'
)

# Write the fixed code back to the file
with open(file_path, 'w') as f:
    f.write(source_code)

print("✅ Source code successfully patched! The bug has been neutralized.")

✅ Source code successfully patched! The bug has been neutralized.


In [11]:
!python train_gan.py --config_path ./configs/train/SRGAN_x4-ImageNet.yaml

2026-03-17 00:28:36.447801: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773707316.469290     209 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773707316.475827     209 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773707316.494428     209 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773707316.494455     209 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773707316.494459     209 computation_placer.cc:177] computation placer alr